2️⃣ ChatPromptTemplate：多角色聊天模板

场景：ChatGPT / Qwen / Claude，一般推荐直接用这个

2.1 单模板字符串版本（简单粗暴）

In [ ]:
from langchain.prompts.chat import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
llm = ChatOpenAI(
    api_key="sk-ruourlqqajhathtbsgmywvanbywoikkwkyhczzqemhkrxcuu",
    base_url="https://api.siliconflow.cn/v1",  # 注意是 base_url
    model="Qwen/Qwen2.5-VL-72B-Instruct",  # 和你 payload 里的 model 保持一致
    temperature=0.7,
    streaming=False,

)
chat_prompt = ChatPromptTemplate.from_template("""
你是一个中文写作助手，请将下面的内容改写得更口语化但保持原意不变：
内容：{text}
""")

parser = StrOutputParser()

# LCEL：prompt -> llm -> parser
chain = chat_prompt | llm | parser

resp = chain.invoke({"text": "本系统主要用于火焰与烟雾的实时检测与告警。"})
print(resp)

2.2 多条消息版本（system + human）

In [ ]:
from langchain.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

system_prompt = SystemMessagePromptTemplate.from_template(
    "你是一个专业的 CV 算法工程师，用通俗语言回答问题。"
)
human_prompt = HumanMessagePromptTemplate.from_template(
    "请解释一下什么是 YOLO，并举一个实际应用场景。"
)

chat_prompt = ChatPromptTemplate.from_messages([
    system_prompt,
    human_prompt,
])

messages = chat_prompt.format_messages()
for m in messages:
    print(m.type, ":", m.content)

✅ 技巧：复用一个 ChatPromptTemplate 做多轮 QA

把 question 做成变量即可：

In [ ]:
qa_chat_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("你是一个友好的技术助手。"),
    HumanMessagePromptTemplate.from_template("问题：{question}")
])

def ask(question: str):
    msgs = qa_chat_prompt.format_messages(question=question)
    return llm.invoke(msgs).content

print(ask("什么是向量数据库？"))
print(ask("Milvus 和 Faiss 有什么区别？"))

3️⃣ ChatMessagePromptTemplate：单条消息模板（通用角色）

场景：自定义角色名 / 动态插入多条消息

In [ ]:
from langchain.prompts.chat import ChatMessagePromptTemplate

# 自定义一个 user 角色消息模板
user_msg_tmpl = ChatMessagePromptTemplate.from_template(
    role="user",
    template="你好，我想了解一下 {topic}"
)

msg = user_msg_tmpl.format(topic="LangChain PromptTemplate 的用法")
print(msg)
# -> ChatMessage(role='user', content='你好，我想了解一下 LangChain PromptTemplate 的用法')

✅ 技巧：动态构造“对话历史列表”

In [ ]:
from langchain.prompts.chat import ChatPromptTemplate

system_msg = ChatMessagePromptTemplate.from_template(
    role="system",
    template="你是一个耐心的 Python 老师。"
)
history_user = ChatMessagePromptTemplate.from_template(
    role="user",
    template="我对 Python 不太熟，你讲解要慢一点。"
)
history_ai = ChatMessagePromptTemplate.from_template(
    role="assistant",
    template="好的，我会尽量用简单的例子来讲解。"
)
current_user = ChatMessagePromptTemplate.from_template(
    role="user",
    template="请问什么是生成器？"
)

chat_prompt = ChatPromptTemplate.from_messages([
    system_msg,
    history_user,
    history_ai,
    current_user,
])

msgs = chat_prompt.format_messages()
print(llm.invoke(msgs).content)
for m in msgs:
    print(f"{m.type}: {m.content}")

4️⃣ SystemMessagePromptTemplate / HumanMessagePromptTemplate / AIMessagePromptTemplate
场景：标准 system / user / assistant 三种角色，语义更清晰

In [ ]:
from langchain.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
)

system_prompt = SystemMessagePromptTemplate.from_template(
    "你是一个资深的运维工程师，专门排查 Linux 和 Docker 问题。"
)
human_history = HumanMessagePromptTemplate.from_template(
    "我服务器磁盘快满了，df -h 显示 overlay2 占了很多空间。"
)
ai_history = AIMessagePromptTemplate.from_template(
    "通常是 Docker 的镜像层和容器层没有清理，可以先看 docker system df。"
)
current_question = HumanMessagePromptTemplate.from_template(
    "那我可以直接删除 overlay2 目录吗？"
)

chat_prompt = ChatPromptTemplate.from_messages([
    system_prompt,
    human_history,
    ai_history,
    current_question,
])

msgs = chat_prompt.format_messages()
for m in msgs:
    print(m.type, ":", m.content)